In [2]:
import pandas as pd

In [3]:
# load datasets
sales_long = pd.read_parquet('../data/processed/01_sales_long.parquet')
calendar = pd.read_csv('../data/raw/calendar.csv')

In [10]:
sales_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 8 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   id        str   
 1   item_id   str   
 2   dept_id   str   
 3   cat_id    str   
 4   store_id  str   
 5   state_id  str   
 6   day       str   
 7   sales     uint16
dtypes: str(7), uint16(1)
memory usage: 6.9 GB


In [6]:
sales_long['sales'].max()

np.int64(763)

changing the datatype of sales column from int64 to uint16 after checking the max and min value of sales to reduce memory usage

In [9]:

sales_long['sales'] = sales_long['sales'].astype('uint16')

Merging the sales long dataset with calender on day(d_x) will give us each day sales along with that day details from calender table. i used left merge to ensure every day sales are recorded. while the data itself has no days skipped left right merge doesnt make any much differnce 

In [25]:
df1 = pd.merge(sales_long, calendar, how='left', left_on='day', right_on='d')

In [13]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 22 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   id            str   
 1   item_id       str   
 2   dept_id       str   
 3   cat_id        str   
 4   store_id      str   
 5   state_id      str   
 6   day           str   
 7   sales         uint16
 8   date          str   
 9   wm_yr_wk      int64 
 10  weekday       str   
 11  wday          int64 
 12  month         int64 
 13  year          int64 
 14  d             str   
 15  event_name_1  str   
 16  event_type_1  str   
 17  event_name_2  str   
 18  event_type_2  str   
 19  snap_CA       int64 
 20  snap_TX       int64 
 21  snap_WI       int64 
dtypes: int64(7), str(14), uint16(1)
memory usage: 14.3 GB


In [26]:
df1[cols].agg(['min', 'max']).T

,min,max
wm_yr_wk,11101,11613
wday,1,7
month,1,12
year,2011,2016
snap_CA,0,1
snap_TX,0,1
snap_WI,0,1


further reducing the memory usage by changing the datatype of int64 to smaller values after checking the min and max limits and chagning the necessary column to date type for better analysis

In [29]:
df1['date'] = pd.to_datetime(df1['date'])

In [27]:
cols = ['wm_yr_wk',"wday", "month", "year", "snap_CA", "snap_TX", "snap_WI"]

for col in cols:
    df1[col] = df1[col].astype('uint16')

In [30]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 22 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            str           
 1   item_id       str           
 2   dept_id       str           
 3   cat_id        str           
 4   store_id      str           
 5   state_id      str           
 6   day           str           
 7   sales         uint16        
 8   date          datetime64[us]
 9   wm_yr_wk      uint16        
 10  weekday       str           
 11  wday          uint16        
 12  month         uint16        
 13  year          uint16        
 14  d             str           
 15  event_name_1  str           
 16  event_type_1  str           
 17  event_name_2  str           
 18  event_type_2  str           
 19  snap_CA       uint16        
 20  snap_TX       uint16        
 21  snap_WI       uint16        
dtypes: datetime64[us](1), str(13), uint16(8)
memory usage: 11.5 GB


after checking te dataset and redcuing the memory usage from 17gb to 11.gb. saving the final sale calender merged dataset in parquet formate

In [35]:
# export to parquet
df1.to_parquet('../data/processed/02_sales_calendar.parquet', index=False)